In [5]:


from getpass import getpass

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document



GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")


with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


splitter = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(raw_text)

documents = [
    Document(page_content=chunk)
    for chunk in chunks
]

print("Total chunks:", len(documents))


embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)


vectorstore = FAISS.from_documents(
    documents,
    embedding=embedding
)

print("FAISS Vector Store created successfully!")



llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)

print("Gemini LLM initialized successfully!")


Enter Gemini API Key: ··········
Total chunks: 1
FAISS Vector Store created successfully!
Gemini LLM initialized successfully!


In [2]:
%pip install -q --no-deps langchain-google-genai==4.4.3

In [3]:
%pip install -q langchain-community==0.4.2 faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
%pip install -q filetype

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.3 requires google-genai>=2.20.0, but you have google-genai 2.12.1 which is incompatible.
langchain-google-genai 4.4.3 requires langchain-core<2.0.0,>=1.7.5, but you have langchain-core 1.7.4 which is incompatible.


In [7]:

# 1) Retrieve-only contextual compression
# Retrieves documents and compresses/filters them.
# It does not generate a final answer.

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using Gemini LLM
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval
print("Contextual Compression Results:")

results = compression_retriever.invoke(
    "Who created LangChain?"
)

for doc in results:
    print("-", doc.page_content)


Contextual Compression Results:


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


- LangChain was created by Harrison Chase.


In [8]:

# 2) Conversational RAG with memory — without agent mode

# Store conversation history
chat_history: list[tuple[str, str]] = []  # [(question, answer), ...]

def rag_chain_fn(question: str) -> str:

    # Retrieve relevant compressed documents
    docs = compression_retriever.invoke(question)

    context = "\n".join(
        doc.page_content for doc in docs
    )

    # Prepare conversation history
    history_text = "\n".join(
        f"Q: {q}\nA: {a}" for q, a in chat_history
    )

    # Create prompt
    prompt = (
        "Answer the question using the conversation history "
        "and context below.\n\n"
        f"Conversation history:\n{history_text}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    # Generate answer using Gemini
    answer = llm.invoke(prompt).content

    # Save conversation history
    chat_history.append((question, answer))

    return answer


# Run conversational RAG
print("Conversational RAG (no agent):")

print(rag_chain_fn("What is LangChain?"))

print(rag_chain_fn("Who created it?"))


Conversational RAG (no agent):


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided context, LangChain is a framework for building applications with LLMs.', 'extras': {'signature': 'EnsKeQFpFH0TWmmrkJBJlA2R1pkSitw6cFyOrLPvqE9q4cY6lROL95iXZSOz5UumaEeGEAuWfmKiUk7C6VhCEVbCvlG2esQq9P+7nu12l6+0296OpdcuiwxjQhID+whn6VanQY8km3t0tPpC8zENYsdPyBTh7Fr5/Ji2gPU='}}]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided context, LangChain was created by Harrison Chase.', 'extras': {'signature': 'EnsKeQFpFH0TDNPnjoWIgKNvafdjV9JswWqn08s58YDg6ccGS1HvARZ31wsc8ZMFFjZoobF4wCi5cKh0RGgs/IK+PhCSNvYG2nle/TJgTLsWSTbfdRTUQaPnVZYOFxUdqTQ+01/Vl5IYPlpFuPwOx9W2ywQuZg+jEkqDLWA='}}]


In [9]:

# 3) Integration into an agent (with a tool)

from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


@tool
def RAG_Tool(question: str) -> str:
    """Answer LangChain-related questions with context."""

    # Retrieve compressed documents
    docs = compression_retriever.invoke(question)

    context = "\n".join(
        doc.page_content for doc in docs
    )

    # Generate answer using Gemini
    answer = llm.invoke(
        f"""
        Answer the question using only the context below.

        Context:
        {context}

        Question:
        {question}
        """
    )

    return answer.content


# Create checkpointer
checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "compression-demo-1"
    }
}


# Create agent
agent = create_agent(
    model=llm,
    tools=[RAG_Tool],
    checkpointer=checkpointer,
)


# Run agent conversation
print("\nAgent Conversation:")

res1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is LangChain?"}
        ]
    },
    config=thread_config
)

print(res1["messages"][-1].content)


res2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Who created it?"}
        ]
    },
    config=thread_config
)

print(res2["messages"][-1].content)



Agent Conversation:


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

[{'type': 'text', 'text': '**LangChain** is an open-source framework designed to simplify the creation of applications using Large Language Models (LLMs). It provides a standard set of interfaces, components, and integrations that connect LLMs to external data sources, APIs, and computation systems, allowing developers to build robust, context-aware AI applications (such as chatbots, question-answering systems, and autonomous agents).', 'extras': {'signature': 'EnsKeQFpFH0T/My8wra/8uv6z+kQr/8mPGYGbY2nJ5H3MP/cwMWi6Ik0syLyydkWLjmQ0laynrPgjOmqcYRy0GbJVfIqv13yenDfIWhx74oafS9wbFZCm8XggNrOlQXAbM2MuUC1SmEbBdPbxQjz3R2gVaQ6JcUwZBFJ9lU='}}]


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

[{'type': 'text', 'text': 'LangChain was created by **Harrison Chase** in October 2022 while he was working at robust intelligence. It quickly grew into an open-source project with contributions from hundreds of developers before the formation of LangChain, Inc. as a company.', 'extras': {'signature': 'EnsKeQFpFH0TviGrbNg7xilxrjC131IcDgAeAswqObGHqIqzDYJ3V/Mh0zk8qpEUmNKsS3VO4BEuAoYoTz+1Gibnk3eZViTYHP/MV2/9KtPEYTWMXp57kWsO7xFqK3JbN156jwuqW6wDSaYyv+0UUcbV28LJ+/rPLpPGKY8='}}]


In [10]:

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Retrieve + generate
query = "Tell me about LangChain creator and features."

source_docs = multi_query_retriever.invoke(query)
context = "\n".join(doc.page_content for doc in source_docs)

answer = llm.invoke(
    f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {query}
    """
).content

print("\nRAG Pipeline (MultiQueryRetriever):")
print("Answer:", answer)

print("\nSources:")
for doc in source_docs:
    print("-", doc.page_content[:200])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



RAG Pipeline (MultiQueryRetriever):
Answer: [{'type': 'text', 'text': 'Based on the provided context:\n\n* **Creator:** LangChain was created by Harrison Chase.\n* **Features:** LangChain supports RAG, agents, memory, tools, and more.', 'extras': {'signature': 'EnsKeQFpFH0TLWdroZaBeMYW4jT/cONOMWLw6alzASwglVi0Ps0qG+FdnmfCUNERD4PB9iQsu0/ZAky5q2TRKi93dxxBEtMan3GGvp58kSvisrln4xjayKwL4EPp6l3XckBnFrsInkE1NQybsqfYsM9Mv/UI+agGbOz+MKU='}}]

Sources:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&
